# Train HARNN — Word2Vec + Global

Paper NLPIR 2023 · Word2Vec self-trained · Global approach

| Cell | Nhiệm vụ |
|------|----------|
| 1 | Import, GPU, load data |
| 2 | Dataset & DataLoader |
| 3 | Train Word2Vec → embedding matrix |
| 4 | HARNN model |
| 5 | Training loop |
| 6 | Đánh giá test set |
| 7 | So sánh với paper + learning curve |


## Cell 1 — Import, GPU, load data


In [ ]:
import json, time, warnings
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────
DATA_DIR   = Path(r'C:\Users\Admin\Documents\nlp\harnn_project\data\process_data')
OUTPUT_DIR = Path(r'C:\Users\Admin\Documents\nlp\harnn_project\output')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for d in ['models', 'models/checkpoints', 'results', 'figures', 'log']:
    (OUTPUT_DIR / d).mkdir(parents=True, exist_ok=True)

# ── GPU ──────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Load data ────────────────────────────────────────────────────────────
with open(DATA_DIR / 'dataset.json',   encoding='utf-8') as f: articles  = json.load(f)
with open(DATA_DIR / 'vocab.json',     encoding='utf-8') as f: vocab     = json.load(f)
with open(DATA_DIR / 'label_map.json', encoding='utf-8') as f: label_map = json.load(f)

VOCAB_SIZE  = len(vocab)
NUM_CLASSES = [len(label_map['l1']), len(label_map['l2']), len(label_map['l3'])]

print(f'\nDataset : {len(articles):,} bài')
print(f'Vocab   : {VOCAB_SIZE:,} từ')
print(f'Classes : L1={NUM_CLASSES[0]}  L2={NUM_CLASSES[1]}  L3={NUM_CLASSES[2]}')


## Cell 2 — Dataset & DataLoader


In [ ]:
MAX_LEN    = 512
BATCH_SIZE = 16
SEED       = 42

class VnExpressDataset(Dataset):
    def __init__(self, articles, vocab, max_len=MAX_LEN):
        self.articles, self.vocab, self.max_len = articles, vocab, max_len

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        a   = self.articles[idx]
        ids = [self.vocab.get(t, 1) for t in a['tokens']][:self.max_len]
        ids += [0] * (self.max_len - len(ids))
        return (
            torch.tensor(ids,        dtype=torch.long),
            torch.tensor(a['vec_l1'], dtype=torch.float),
            torch.tensor(a['vec_l2'], dtype=torch.float),
            torch.tensor(a['vec_l3'], dtype=torch.float),
        )

dataset = VnExpressDataset(articles, vocab)
n       = len(dataset)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
n_test  = n - n_train - n_val

gen = torch.Generator().manual_seed(SEED)
train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test], generator=gen)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,}')
print(f'Batches/epoch: {len(train_loader)}')

ids, l1, l2, l3 = next(iter(train_loader))
print(f'Batch: ids={tuple(ids.shape)}  l1={tuple(l1.shape)}  l2={tuple(l2.shape)}  l3={tuple(l3.shape)}')


## Cell 3 — Train Word2Vec → embedding matrix
Tham số theo paper: `vector_size=100`, `min_count=5`, `epochs=10`, Skip-gram.


In [ ]:
from gensim.models import Word2Vec

EMBED_DIM = 100

corpus = [a['tokens'] for a in articles]
print(f'Corpus: {len(corpus):,} bài  {sum(len(t) for t in corpus):,} tokens')

t0  = time.time()
w2v = Word2Vec(
    sentences=corpus, vector_size=EMBED_DIM,
    min_count=2, epochs=10, sg=1, workers=4, seed=SEED,
)
w2v.save(str(OUTPUT_DIR / 'models' / 'word2vec.model'))
print(f'W2V vocab: {len(w2v.wv):,} từ  ({time.time()-t0:.1f}s)')

# Embedding matrix để nạp vào nn.Embedding
embed_matrix    = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
embed_matrix[1] = w2v.wv.vectors.mean(axis=0)  # UNK = mean vector

found = sum(1 for w, i in vocab.items() if w in w2v.wv
            and not embed_matrix.__setitem__(i, w2v.wv[w]))
print(f'Coverage: {found:,}/{VOCAB_SIZE-2:,} ({found/(VOCAB_SIZE-2)*100:.1f}%)')

test_word = 'bóng_đá'
if test_word in w2v.wv:
    print(f"Từ gần '{test_word}': {[w for w,_ in w2v.wv.most_similar(test_word, topn=5)]}")


## Cell 4 — HARNN model
`DRL` (Embedding + BiGRU) → `HARL` (Attention × 3) → `HAM` (LSTMCell) → `HPL` (Linear + Sigmoid)


In [ ]:
class HARNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size,
                 num_classes_per_level, pretrained_embeddings=None, dropout=0.5):
        super().__init__()
        self.num_levels  = len(num_classes_per_level)
        self.hidden_size = hidden_size

        # DRL
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained_embeddings))
        self.bigru   = nn.GRU(embed_dim, hidden_size, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)

        # HARL: 1 attention layer per level
        self.attention   = nn.ModuleList([nn.Linear(hidden_size * 2, 1) for _ in range(self.num_levels)])

        # HAM
        self.ham         = nn.LSTMCell(hidden_size * 2, hidden_size)

        # HPL: context(hidden*2) + ham(hidden) = hidden*3
        self.classifiers = nn.ModuleList([nn.Linear(hidden_size * 3, n) for n in num_classes_per_level])

    def forward(self, x):
        B      = x.size(0)
        emb    = self.dropout(self.embedding(x))
        doc, _ = self.bigru(emb)
        doc    = self.dropout(doc)

        h = torch.zeros(B, self.hidden_size, device=x.device)
        c = torch.zeros(B, self.hidden_size, device=x.device)

        preds = []
        for lv in range(self.num_levels):
            context = (torch.softmax(self.attention[lv](doc), dim=1) * doc).sum(dim=1)
            h, c    = self.ham(context, (h, c))
            feat    = self.dropout(torch.cat([context, h], dim=-1))
            preds.append(torch.sigmoid(self.classifiers[lv](feat)))

        return preds


HIDDEN_SIZE = 256
model = HARNN(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_size=HIDDEN_SIZE,
    num_classes_per_level=NUM_CLASSES, pretrained_embeddings=embed_matrix, dropout=0.5,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Params: {n_params:,}')
print(f'Output shapes: {[tuple(o.shape) for o in model(torch.zeros(2, MAX_LEN, dtype=torch.long).to(device))]}')


## Cell 5 — Training loop


In [ ]:
NUM_EPOCHS    = 20
LR            = 1e-3
PATIENCE      = 5
LEVEL_WEIGHTS = [1.0, 1.0, 0.8]
CKPT_PATH     = OUTPUT_DIR / 'models' / 'checkpoints' / 'best_model.pt'

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)
criterion = nn.BCELoss()


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0
    all_preds  = [[] for _ in range(3)]
    all_labels = [[] for _ in range(3)]

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for ids, l1, l2, l3 in loader:
            ids     = ids.to(device)
            targets = [l1.to(device), l2.to(device), l3.to(device)]
            if training: optimizer.zero_grad()

            preds = model(ids)
            loss  = sum(LEVEL_WEIGHTS[i] * criterion(preds[i], targets[i])
                        for i in range(3) if targets[i].sum() > 0)

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            total_loss += loss.item()
            for i in range(3):
                all_preds[i].append(preds[i].detach().cpu().numpy())
                all_labels[i].append(targets[i].cpu().numpy())

    f1s = []
    for i in range(3):
        y = np.concatenate(all_labels[i])
        p = np.concatenate(all_preds[i])
        f1s.append(f1_score(y, (p >= 0.5).astype(int), average='micro', zero_division=0)
                   if y.sum() > 0 else 0.0)
    return total_loss / len(loader), f1s


history        = {'train_loss': [], 'val_loss': [], 'val_f1_l1': [], 'val_f1_l2': [], 'val_f1_l3': []}
best_f1        = 0.0
no_improve     = 0

print(f'Train {NUM_EPOCHS} epochs | {device} | batch={BATCH_SIZE}\n')
print(f'{"Epoch":>5} | {"Train":>8} | {"Val":>8} | {"F1-L1":>6} | {"F1-L2":>6} | {"F1-L3":>6} | {"Time":>5}')
print('-' * 62)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, _       = run_epoch(train_loader, training=True)
    val_loss,   val_f1s = run_epoch(val_loader,   training=False)

    for k, v in zip(['train_loss','val_loss','val_f1_l1','val_f1_l2','val_f1_l3'],
                    [train_loss, val_loss, *val_f1s]):
        history[k].append(v)

    print(f'{epoch:>5} | {train_loss:>8.4f} | {val_loss:>8.4f} | '
          f'{val_f1s[0]:>6.3f} | {val_f1s[1]:>6.3f} | {val_f1s[2]:>6.3f} | {time.time()-t0:>4.1f}s')

    scheduler.step(val_f1s[0])
    global_f1 = (val_f1s[0] + val_f1s[1] + val_f1s[2]) / 3
    if global_f1 > best_f1:
        best_f1 = global_f1
        torch.save({'epoch': epoch,
                'model_state': model.state_dict(),
                'val_f1_l1': val_f1s[0],
                'val_f1s': val_f1s,
                'global_f1': global_f1}, CKPT_PATH)
        print(f'       ✓ Saved (Global F1={global_f1:.3f}  L1={val_f1s[0]:.3f}  L2={val_f1s[1]:.3f}  L3={val_f1s[2]:.3f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping epoch {epoch}')
            break

json.dump(history, open(OUTPUT_DIR / 'results' / 'train_history.json', 'w'), indent=2)
print(f'\nBest F1-L1: {best_f1:.3f}')


## Cell 6 — Đánh giá test set


In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Checkpoint epoch {ckpt['epoch']}  val F1-L1={ckpt['val_f1_l1']:.3f}")


@torch.no_grad()
def evaluate(loader):
    preds_list  = [[] for _ in range(3)]
    labels_list = [[] for _ in range(3)]
    for ids, l1, l2, l3 in loader:
        out = model(ids.to(device))
        for i, lbl in enumerate([l1, l2, l3]):
            preds_list[i].append(out[i].cpu().numpy())
            labels_list[i].append(lbl.numpy())
    return ([np.concatenate(p) for p in preds_list],
            [np.concatenate(l) for l in labels_list])


def compute_metrics(preds, labels, threshold=0.5):
    valid = labels.sum(axis=1) > 0
    if not valid.any():
        return dict(precision=0, recall=0, f1=0, auprc=0)
    pb = (preds >= threshold).astype(int)
    return dict(
        precision = precision_score(labels[valid], pb[valid], average='micro', zero_division=0),
        recall    = recall_score   (labels[valid], pb[valid], average='micro', zero_division=0),
        f1        = f1_score       (labels[valid], pb[valid], average='micro', zero_division=0),
        auprc     = average_precision_score(labels[valid], preds[valid], average='micro'),
    )


preds_list, labels_list = evaluate(test_loader)
results = {name: compute_metrics(preds_list[i], labels_list[i])
           for i, name in enumerate(['L1', 'L2', 'L3'])}

all_m = list(results.values())
results['Global'] = {k: float(np.mean([m[k] for m in all_m])) for k in ['precision','recall','f1']}

print(f'\n{"="*60}')
print(f'  KẾT QUẢ TEST SET — Word2Vec + Global')
print(f'{"="*60}')
print(f'  {"Level":<8} {"Precision":>10} {"Recall":>8} {"F1":>8} {"AUPRC":>8}')
print(f'  {"-"*50}')
for name in ['L1', 'L2', 'L3']:
    m = results[name]
    print(f'  {name:<8} {m["precision"]:>10.3f} {m["recall"]:>8.3f} {m["f1"]:>8.3f} {m["auprc"]:>8.3f}')
print(f'  {"-"*50}')
g = results['Global']
print(f'  {"Global":<8} {g["precision"]:>10.3f} {g["recall"]:>8.3f} {g["f1"]:>8.3f}')
print(f'{"="*60}')

json.dump(results, open(OUTPUT_DIR / 'results' / 'results_w2v_global.json', 'w'), indent=2)


## Cell 7 — learning curve


In [ ]:
# Learning curve
import matplotlib.pyplot as plt

history = json.load(open(OUTPUT_DIR / "results" / "train_history.json"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"],   label="Val")
axes[0].set_title("Loss");  axes[0].legend()

axes[1].plot(history["val_f1_l1"], label="F1 L1")
axes[1].plot(history["val_f1_l2"], label="F1 L2")
axes[1].plot(history["val_f1_l3"], label="F1 L3")
axes[1].set_title("Val F1 theo level");  axes[1].legend()

plt.tight_layout()
fig_path = OUTPUT_DIR / "figures" / "learning_curve.png"
plt.savefig(fig_path, dpi=120)
plt.show()
print(f"Saved: {fig_path}")
